# **Preprocessing Eye**

In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


class Config:
    
    def __init__(self):
        self.root_dir = "/content/dataset1"
        self.save_root = "/content/preprocessed_eye"

        self.split = "train"
        self.region = "right_eye"

        self.model_path = "/content/face_landmarker.task"
        self.num_faces = 1
        self.min_face_detection_confidence = 0.2
        self.min_face_presence_confidence = 0.2
        self.min_tracking_confidence = 0.2

        self.padding_ratio = 0.40
        self.min_eye_size = 45

        self.gray_fill_value = 128
        self.save_ext = ".png"

        self.enable_adaptive_smoothing = True
        self.motion_scale = 20.0

        self.fail_if_label_missing = True
        self.normalize_output_condition = True


class MediaPipeTasksDetector:
    def __init__(
        self,
        model_path,
        num_faces=1,
        min_face_detection_confidence=0.2,
        min_face_presence_confidence=0.2,
        min_tracking_confidence=0.2,
    ):
        base_options = python.BaseOptions(model_asset_path=model_path)
        options = vision.FaceLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            num_faces=num_faces,
            min_face_detection_confidence=min_face_detection_confidence,
            min_face_presence_confidence=min_face_presence_confidence,
            min_tracking_confidence=min_tracking_confidence,
            output_face_blendshapes=False,
            output_facial_transformation_matrixes=False,
        )
        self.detector = vision.FaceLandmarker.create_from_options(options)

        self.left_eye_indices = [33, 133, 160, 159, 158, 157, 173, 246, 161, 144, 145, 153, 154, 155]
        self.right_eye_indices = [362, 263, 387, 386, 385, 384, 398, 466, 388, 373, 374, 380, 381, 382]

    def close(self):
        self.detector.close()

    def detect(self, frame_bgr):
        h, w = frame_bgr.shape[:2]

        try:
            if len(frame_bgr.shape) == 2:
                rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_GRAY2RGB)
            else:
                rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = self.detector.detect(mp_image)
        except Exception:
            return None

        if not result.face_landmarks:
            return None

        face_landmarks = result.face_landmarks[0]

        def point(idx):
            lm = face_landmarks[idx]
            return float(lm.x * w), float(lm.y * h)

        left_eye_points = [point(i) for i in self.left_eye_indices]
        right_eye_points = [point(i) for i in self.right_eye_indices]

        left_eye_center = (
            float(np.mean([p[0] for p in left_eye_points])),
            float(np.mean([p[1] for p in left_eye_points])),
        )
        right_eye_center = (
            float(np.mean([p[0] for p in right_eye_points])),
            float(np.mean([p[1] for p in right_eye_points])),
        )

        xs = [p[0] for p in left_eye_points + right_eye_points]
        ys = [p[1] for p in left_eye_points + right_eye_points]

        return {
            "facial_area": [min(xs), min(ys), max(xs), max(ys)],
            "landmarks": {
                "left_eye": left_eye_center,
                "right_eye": right_eye_center,
                "left_eye_points": left_eye_points,
                "right_eye_points": right_eye_points,
            },
        }


def standardize_condition_name(condition):
    condition = condition.strip().lower()
    mapping = {
        "nightnoglasses": "nightnoglasses",
        "night_no_glasses": "nightnoglasses",
        "night-noglasses": "nightnoglasses",
        "night_noglasses": "nightnoglasses",
        "noglasses": "noglasses",
        "no_glasses": "noglasses",
        "nightglasses": "nightglasses",
        "glasses": "glasses",
        "sunglasses": "sunglasses",
    }
    return mapping.get(condition, condition)


def build_output_condition(condition, cfg):
    if cfg.normalize_output_condition:
        return standardize_condition_name(condition)
    return condition


def label_suffix_for_region(region):
    region = region.lower()
    if region in {"left_eye", "right_eye"}:
        return "eye"
    raise ValueError(f"Unsupported region: {region}")


def parse_validation_or_test_video_name(video_stem):
    parts = video_stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Invalid filename format: {video_stem}")

    person_id = parts[0]
    state = parts[-1]
    raw_condition = "_".join(parts[1:-1])
    condition = standardize_condition_name(raw_condition)

    return person_id, condition, state


def load_labels(label_path):
    if not os.path.isfile(label_path):
        raise FileNotFoundError(label_path)

    with open(label_path, "r") as f:
        content = f.read().strip()

    return [int(ch) for ch in content if ch.isdigit()]


def get_video_fps(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 30

    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()

    if fps is None or fps <= 0:
        return 30

    return int(round(fps))


def ensure_gray(frame):
    if len(frame.shape) == 2:
        return frame
    return cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)


def crop_with_replicate_padding(img, x1, y1, x2, y2):
    h, w = img.shape[:2]

    pad_left = max(0, -x1)
    pad_top = max(0, -y1)
    pad_right = max(0, x2 - w)
    pad_bottom = max(0, y2 - h)

    if pad_left or pad_top or pad_right or pad_bottom:
        img = cv2.copyMakeBorder(
            img,
            pad_top,
            pad_bottom,
            pad_left,
            pad_right,
            borderType=cv2.BORDER_REPLICATE
        )

    x1 += pad_left
    x2 += pad_left
    y1 += pad_top
    y2 += pad_top

    return img[y1:y2, x1:x2]


def compute_square_eye_box(eye_points, padding_ratio, min_size):
    xs = [p[0] for p in eye_points]
    ys = [p[1] for p in eye_points]

    x_min = min(xs)
    x_max = max(xs)
    y_min = min(ys)
    y_max = max(ys)

    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0

    eye_w = x_max - x_min
    eye_h = y_max - y_min
    size = max(eye_w, eye_h)

    size = size * (1.0 + padding_ratio)
    size = max(size, float(min_size))

    half_size = size / 2.0

    x1 = center_x - half_size
    y1 = center_y - half_size
    x2 = center_x + half_size
    y2 = center_y + half_size

    return [x1, y1, x2, y2]


def smooth_box(prev_box, curr_box, motion_scale=20.0):
    if prev_box is None:
        return [float(v) for v in curr_box]

    prev = np.array(prev_box, dtype=np.float32)
    curr = np.array(curr_box, dtype=np.float32)

    movement = np.linalg.norm(curr - prev)
    motion_factor = min(1.0, movement / motion_scale)

    alpha = 0.7 - 0.4 * motion_factor
    smoothed = alpha * prev + (1.0 - alpha) * curr

    return smoothed.astype(np.float32).tolist()


def make_gray_from_box(box, gray_fill_value, min_eye_size):
    if box is not None:
        x1, y1, x2, y2 = box
        h = max(1, int(round(y2 - y1)))
        w = max(1, int(round(x2 - x1)))
        return np.full((h, w), gray_fill_value, dtype=np.uint8)

    size = max(1, int(round(min_eye_size)))
    return np.full((size, size), gray_fill_value, dtype=np.uint8)


class EyePreprocessor:
    def __init__(self, cfg):
        self.region = cfg.region
        self.padding_ratio = cfg.padding_ratio
        self.min_eye_size = cfg.min_eye_size
        self.enable_adaptive_smoothing = cfg.enable_adaptive_smoothing
        self.motion_scale = cfg.motion_scale
        self.gray_fill_value = cfg.gray_fill_value

        self.prev_real_box = None
        self.prev_valid_crop = None
        self.prev_valid_box = None

        self.prev_crop_used_consecutively = 0
        self.prev_position_used_consecutively = 0

    def reset_video_state(self):
        self.prev_real_box = None
        self.prev_valid_crop = None
        self.prev_valid_box = None
        self.prev_crop_used_consecutively = 0
        self.prev_position_used_consecutively = 0

    def get_limits(self, fps):
        if int(round(fps)) == 30:
            return 4, 18
        return 2, 9

    def extract_valid_crop(self, frame, face):
        if face is None or "landmarks" not in face:
            return None

        landmarks = face["landmarks"]
        eye_points = landmarks["left_eye_points"] if self.region == "left_eye" else landmarks["right_eye_points"]

        raw_box = compute_square_eye_box(
            eye_points,
            self.padding_ratio,
            self.min_eye_size
        )

        if self.enable_adaptive_smoothing:
            box = smooth_box(self.prev_real_box, raw_box, self.motion_scale)
        else:
            box = raw_box

        x1, y1, x2, y2 = [int(round(v)) for v in box]

        gray_frame = ensure_gray(frame)
        crop = crop_with_replicate_padding(gray_frame, x1, y1, x2, y2)

        if crop is None or crop.size == 0:
            return None

        box_int = [x1, y1, x2, y2]

        self.prev_real_box = box_int
        self.prev_valid_crop = crop.copy()
        self.prev_valid_box = box_int
        self.prev_crop_used_consecutively = 0
        self.prev_position_used_consecutively = 0

        return crop, 1, "detected"

    def extract_with_fallback(self, frame, face, fps):
        prev_crop_limit, prev_pos_limit = self.get_limits(fps)

        valid_result = self.extract_valid_crop(frame, face)
        if valid_result is not None:
            return valid_result

        if self.prev_valid_crop is not None and self.prev_crop_used_consecutively < prev_crop_limit:
            self.prev_crop_used_consecutively += 1
            self.prev_position_used_consecutively = 0
            return self.prev_valid_crop.copy(), 0, "prev_crop"

        if self.prev_valid_box is not None and self.prev_position_used_consecutively < prev_pos_limit:
            x1, y1, x2, y2 = self.prev_valid_box
            gray_frame = ensure_gray(frame)
            crop = crop_with_replicate_padding(gray_frame, x1, y1, x2, y2)
            self.prev_position_used_consecutively += 1
            self.prev_crop_used_consecutively = 0
            return crop, 0, "prev_position"

        gray_crop = make_gray_from_box(
            self.prev_valid_box,
            self.gray_fill_value,
            self.min_eye_size
        )
        return gray_crop, 0, "gray"


def process_video(video_path, label_path, output_dir, save_prefix, region, detector, cfg):
    labels = load_labels(label_path)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = get_video_fps(video_path)
    cropper = EyePreprocessor(cfg)
    cropper.reset_video_state()

    reported_total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    actual_frames_read = 0
    frames_saved = 0
    valid_count = 0
    invalid_count = 0

    detected_count = 0
    prev_crop_count = 0
    prev_position_count = 0
    gray_count = 0

    frame_idx = 0

    try:
        while frame_idx < len(labels):
            ret, frame = cap.read()
            if not ret:
                break

            actual_frames_read += 1

            face = detector.detect(frame)
            crop, validity, mode = cropper.extract_with_fallback(frame, face, fps)

            label = labels[frame_idx]
            out_name = f"{save_prefix}_{frame_idx:06d}_{validity}_{label}{cfg.save_ext}"
            out_path = os.path.join(output_dir, out_name)

            ok = cv2.imwrite(out_path, crop)
            if not ok:
                raise RuntimeError(f"Failed to save image: {out_path}")

            frames_saved += 1

            if validity == 1:
                valid_count += 1
            else:
                invalid_count += 1

            if mode == "detected":
                detected_count += 1
            elif mode == "prev_crop":
                prev_crop_count += 1
            elif mode == "prev_position":
                prev_position_count += 1
            elif mode == "gray":
                gray_count += 1

            frame_idx += 1

    finally:
        cap.release()

    prev_crop_limit, prev_pos_limit = cropper.get_limits(fps)

    print("=" * 80)
    print(f"Video                : {os.path.basename(video_path)}")
    print(f"Region               : {region}")
    print(f"Output folder        : {output_dir}")
    print(f"FPS                  : {fps}")
    print(f"Prev crop limit      : {prev_crop_limit}")
    print(f"Prev position limit  : {prev_pos_limit}")
    print(f"Reported video frames: {reported_total_frames}")
    print(f"Actual frames read   : {actual_frames_read}")
    print(f"Total labels         : {len(labels)}")
    print(f"Frames saved         : {frames_saved}")
    print(f"Valid frames         : {valid_count}")
    print(f"Invalid frames       : {invalid_count}")
    print("-" * 80)
    print(f"Detected normally    : {detected_count}")
    print(f"Prev crop reused     : {prev_crop_count}")
    print(f"Prev position        : {prev_position_count}")
    print(f"Gray fallback        : {gray_count}")

    missing_from_video = len(labels) - actual_frames_read
    if missing_from_video > 0:
        print(f"Warning              : video ended early by {missing_from_video} frame(s)")
    elif missing_from_video < 0:
        print(f"Warning              : video has {-missing_from_video} extra frame(s)")
    else:
        print("Status               : labels and readable frames matched")
    print("=" * 80)


def process_train(cfg, detector):
    if not os.path.isdir(cfg.root_dir):
        raise FileNotFoundError(f"Root directory not found: {cfg.root_dir}")

    suffix = label_suffix_for_region(cfg.region)

    for person_entry in sorted(os.scandir(cfg.root_dir), key=lambda e: e.name):
        if not person_entry.is_dir():
            continue

        person_id = person_entry.name

        for condition_entry in sorted(os.scandir(person_entry.path), key=lambda e: e.name):
            if not condition_entry.is_dir():
                continue

            raw_condition = condition_entry.name
            out_condition = build_output_condition(raw_condition, cfg)

            for file_entry in sorted(os.scandir(condition_entry.path), key=lambda e: e.name):
                if not file_entry.is_file() or not file_entry.name.lower().endswith(".avi"):
                    continue

                behavior = os.path.splitext(file_entry.name)[0]
                video_path = file_entry.path
                label_path = os.path.join(condition_entry.path, f"{person_id}_{behavior}_{suffix}.txt")

                if not os.path.isfile(label_path):
                    msg = f"Missing train label: {label_path}"
                    if cfg.fail_if_label_missing:
                        raise FileNotFoundError(msg)
                    print(f"[SKIP] {msg}")
                    continue

                output_dir = os.path.join(cfg.save_root, person_id, out_condition, behavior)
                os.makedirs(output_dir, exist_ok=True)

                save_prefix = f"{person_id}_{out_condition}_{behavior}"

                process_video(
                    video_path=video_path,
                    label_path=label_path,
                    output_dir=output_dir,
                    save_prefix=save_prefix,
                    region=cfg.region,
                    detector=detector,
                    cfg=cfg,
                )


def process_validation(cfg, detector):
    if not os.path.isdir(cfg.root_dir):
        raise FileNotFoundError(f"Root directory not found: {cfg.root_dir}")

    suffix = label_suffix_for_region(cfg.region)

    for person_entry in sorted(os.scandir(cfg.root_dir), key=lambda e: e.name):
        if not person_entry.is_dir():
            continue

        folder_person_id = person_entry.name

        for file_entry in sorted(os.scandir(person_entry.path), key=lambda e: e.name):
            if not file_entry.is_file() or not file_entry.name.lower().endswith((".avi", ".mp4")):
                continue

            video_stem = os.path.splitext(file_entry.name)[0]
            person_id, condition, state = parse_validation_or_test_video_name(video_stem)

            if person_id != folder_person_id:
                raise ValueError(f"Subject mismatch: {file_entry.path}")

            label_path = os.path.join(person_entry.path, f"{person_id}_{condition}_mixing_{suffix}.txt")

            if not os.path.isfile(label_path):
                msg = f"Missing validation label: {label_path}"
                if cfg.fail_if_label_missing:
                    raise FileNotFoundError(msg)
                print(f"[SKIP] {msg}")
                continue

            out_condition = build_output_condition(condition, cfg)
            output_dir = os.path.join(cfg.save_root, person_id, out_condition, state)
            os.makedirs(output_dir, exist_ok=True)
            save_prefix = f"{person_id}_{out_condition}_{state}"

            process_video(
                video_path=file_entry.path,
                label_path=label_path,
                output_dir=output_dir,
                save_prefix=save_prefix,
                region=cfg.region,
                detector=detector,
                cfg=cfg,
            )


def process_test(cfg, detector):
    if not os.path.isdir(cfg.root_dir):
        raise FileNotFoundError(f"Root directory not found: {cfg.root_dir}")

    label_root = os.path.join(cfg.root_dir, "test_label_txt", "wh")
    if not os.path.isdir(label_root):
        raise FileNotFoundError(f"Test label folder not found: {label_root}")

    for entry in sorted(os.scandir(cfg.root_dir), key=lambda e: e.name):
        if not entry.is_file() or not entry.name.lower().endswith((".avi", ".mp4")):
            continue

        video_stem = os.path.splitext(entry.name)[0]
        person_id, condition, state = parse_validation_or_test_video_name(video_stem)
        label_path = os.path.join(label_root, f"{person_id}_{condition}_mixing_drowsiness.txt")

        if not os.path.isfile(label_path):
            msg = f"Missing test label: {label_path}"
            if cfg.fail_if_label_missing:
                raise FileNotFoundError(msg)
            print(f"[SKIP] {msg}")
            continue

        out_condition = build_output_condition(condition, cfg)
        output_dir = os.path.join(cfg.save_root, person_id, out_condition, state)
        os.makedirs(output_dir, exist_ok=True)
        save_prefix = f"{person_id}_{out_condition}_{state}"

        process_video(
            video_path=entry.path,
            label_path=label_path,
            output_dir=output_dir,
            save_prefix=save_prefix,
            region=cfg.region,
            detector=detector,
            cfg=cfg,
        )


def run(cfg):
    os.makedirs(cfg.save_root, exist_ok=True)

    detector = MediaPipeTasksDetector(
        model_path=cfg.model_path,
        num_faces=cfg.num_faces,
        min_face_detection_confidence=cfg.min_face_detection_confidence,
        min_face_presence_confidence=cfg.min_face_presence_confidence,
        min_tracking_confidence=cfg.min_tracking_confidence,
    )

    try:
        split = cfg.split.lower()
        if split == "train":
            process_train(cfg, detector)
        elif split in {"validation", "val"}:
            process_validation(cfg, detector)
        elif split == "test":
            process_test(cfg, detector)
        else:
            raise ValueError(f"Unsupported split: {cfg.split}")
    finally:
        detector.close()


if __name__ == "__main__":
    cfg = Config()

    cfg.root_dir = "/content/dataset1"
    cfg.save_root = "/content/preprocessed_eye_right_nowarp"

    cfg.split = "train"
    cfg.region = "right_eye"

    cfg.model_path = "/content/face_landmarker.task"
    cfg.num_faces = 1
    cfg.min_face_detection_confidence = 0.2
    cfg.min_face_presence_confidence = 0.2
    cfg.min_tracking_confidence = 0.2

    cfg.padding_ratio = 0.40
    cfg.min_eye_size = 45

    cfg.enable_adaptive_smoothing = True
    cfg.motion_scale = 20.0

    cfg.gray_fill_value = 128
    cfg.fail_if_label_missing = True
    cfg.normalize_output_condition = True

    run(cfg)